# За вкъщи, упражнение 1

**Машинно самообучение · зимен семестър 2026/2027**

Пет задачи в реда на упражнението: една върху Python, една върху модули и три
върху NumPy. За всяка има подсказка, която можем да отворим при нужда.
Отговорите на въпросите пишем в текстова (Markdown) клетка.

**Преди предаване:** рестартираме ядрото и изпълняваме всички клетки отначало
докрай. Проверяваме решенията и очакваните резултати, оставяме изхода видим и
включваме текстовите отговори.


In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys
import numpy as np

# Работи и когато Jupyter е стартиран от общата папка на упражненията.
base = Path.cwd()
if not (base / "data" / "housing_prices.csv").exists():
    base = base / "01_Python_Notebooks_NumPy"
data_path = base / "data" / "housing_prices.csv"
assert data_path.exists(), "Стартираме тетрадката от папката на упражнението или от общата папка."
sys.path.insert(0, str(base.resolve()))

homes = np.genfromtxt(data_path, delimiter=",", skip_header=1)
training_homes = homes[:8]
areas = training_homes[:, 0]
prices = training_homes[:, 1]
slope, intercept = np.polyfit(areas, prices, deg=1)
print("учебни имоти:", training_homes.shape, "наклон:", round(slope, 2))

---

## Задача 1: броене на повторения

Да се напише функция `count_occurrences(values)`, която получава списък и
връща речник `{value: count}`. Без `collections.Counter`.

<details>
<summary>Подсказка</summary>

Започваме с празен речник. За всяка стойност обновяваме
`counts[value] = counts.get(value, 0) + 1`.

</details>

In [2]:
values = [50, 55, 50, 60, 55, 50]


def count_occurrences(values):
    # Решение
    ...


# Очаквано: {50: 3, 55: 2, 60: 1}
print(count_occurrences(values))

None


---

## Задача 2: допълване на пакета `utils`

В [`utils/distances.py`](utils/distances.py) се добавя функция:

```python
def manhattan(a, b):
    '''Манхатъново разстояние: сумата от абсолютните разлики.'''
    ...
```

След това `"manhattan"` се добавя в `__all__` и във вноса в [`utils/__init__.py`](utils/__init__.py).
Благодарение на `autoreload` рестарт на ядрото не е нужен.

**Въпрос:** `euclidean([1, 2, 3], [4, 5, 6])` дава около 5.196, а `manhattan` дава 6.0.
Защо са различни и в кой случай се предпочита едното пред другото? Отговор с едно или
две изречения в markdown клетката след кода.

<details>
<summary>Подсказка</summary>

`np.abs` върху разликата, после `np.sum`. Връща се `float`, както в `euclidean`.

</details>

In [3]:
from utils import euclidean

print("евклидово:", euclidean([1, 2, 3], [4, 5, 6]))

try:
    from utils import manhattan
except ImportError:
    print("Добавяме manhattan в utils и изпълняваме клетката отново.")
else:
    print("манхатън: ", manhattan([1, 2, 3], [4, 5, 6]))


евклидово: 5.196152422706632
Добавяме manhattan в utils и изпълняваме клетката отново.


*Отговор:*

---

## Задача 3: мащабиране в интервала [0, 1]

Да се напише функция `minmax(X)`, която мащабира **всяка колона поотделно**
в интервала `[0, 1]` по формулата

$$x' = \frac{x - \min}{\max - \min}$$

Без цикли, чрез `axis=0` и broadcasting. Внимание при колона с еднакви
стойности, при която знаменателят е нула. За тази задача работим с
двете числови колони на `homes`, независимо от ролята им в модела.

Резултатът се сравнява с `utils.minmax_scale` чрез `np.allclose`.

<details>
<summary>Подсказка</summary>

`X.min(axis=0)` и `X.max(axis=0)` дават по една стойност на признак. Знаменателят се пази от нула с `np.where(span == 0, 1.0, span)`.

</details>

In [ ]:
def minmax(X):
    # Решение
    ...


from utils import minmax_scale

# Очаквано: True, и колони с минимум 0 и максимум 1
# print(np.allclose(minmax(homes), minmax_scale(homes)))

---

## Задача 4: предсказване за няколко площи

Написваме функция `predict_price(area_m2, slope, intercept)`, която
използва уравнението на правата. Функцията трябва да приема както едно
число, така и NumPy масив. Предсказваме цени за жилища от `55`, `75` и
`95` m² **с една операция върху масив**.

С колко евро се различават предсказанията за 75 и 95 m²? Свързваме
резултата с наклона `slope`.

<details>
<summary>Подсказка</summary>

Изразът `slope * area_m2 + intercept` работи поелементно и за масив.
Разлика от 20 m² в аргумента дава разлика `20 * slope` в предсказанието.

</details>

In [ ]:
def predict_price(area_m2, slope, intercept):
    # Решение
    ...


query_areas = np.array([55.0, 75.0, 95.0])
# predicted_prices = ...
# print(np.round(predicted_prices, 0))

*Отговор:*

---

## Задача 5: проверка с нови жилища

Имаме две нови двойки площ–цена, които **не са използвани** при
обучението: `(62 m², 158000 евро)` и `(108 m², 248000 евро)`.

Предсказваме двете цени и изчисляваме средната абсолютна грешка.
За сравнение правим постоянна прогноза: за двата имота предсказваме
една и съща цена, независимо от площта им: средната цена от учебните
данни `prices.mean()`. Изчисляваме и нейната
грешка. Кое правило се справя по-добре върху **тези два** имота? Защо
не правим общ извод от толкова малък набор?

<details>
<summary>Подсказка</summary>

За грешката използваме `np.mean(np.abs(actual - predicted))`. За
постоянната прогноза можем да извадим едно число `prices.mean()` от
масива с двете нови цени.

</details>

In [ ]:
test_areas = np.array([62.0, 108.0])
test_prices = np.array([158_000.0, 248_000.0])

# Решение: предсказания, две средни абсолютни грешки и сравнение

*Отговор:*